
========================================================================
NOTEBOOK 01: DATA PREP
========================================================================

Run this notebook ONCE. Outputs go to data/processed/ and are reused
by notebooks 02 and 03.

What this notebook does:
  1. Load the NYT China corpus (5396 articles, 2020-2024)
  2. Build distinctive entity lexicons (BUSINESS_FRAME_V4, WORLD_FRAME_V4)
  3. Sample 20 headlines (10 Business + 10 Foreign), random_state=42,
     excluding Shanghai/Balloon case-study windows
  4. Score the 20 real NYT articles on:
     - Entity axis (lexicon hit rates)
     - Sentiment axis (DistilBERT chunked)
  5. Save outputs to data/processed/

Outputs:
  - data/processed/experiment_sample.csv     # 20 sampled article rows
  - data/processed/lexicons.json              # BUSINESS/WORLD frame V4
  - data/processed/nyt_baseline_axis1.csv     # NYT entity scores
  - data/processed/nyt_baseline_axis2.csv     # NYT sentiment scores

Total runtime: ~5-10 minutes (most of it is DistilBERT scoring).
Cost: $0 (no API calls).

In [ ]:

# =============================================================================
# CELL 1: Setup and configuration
# =============================================================================

import os
import re
import json
import gc
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Path config — relative to this notebook (notebooks/)
ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"ROOT:      {ROOT}")
print(f"RAW:       {RAW}")
print(f"PROCESSED: {PROCESSED}")

# Stable env settings (avoid Apple Silicon quirks)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

# Random seed — must match across notebooks
SEED = 42

# Sample size (expanded)
N_ORIG_PER_DESK  = 10    # original locked sample — keeps sample_id 0–19 stable
N_TOTAL_PER_DESK = 40    # expanded target per desk (40 Business + 40 Foreign = 80)
SEED_EXPAND      = 43    # separate seed for the newly added headlines

# Case-study exclusion windows (reserved for future case-study chapter)
SHANGHAI_START = pd.Timestamp("2022-03-25")
SHANGHAI_END   = pd.Timestamp("2022-04-25")
BALLOON_START  = pd.Timestamp("2023-01-25")
BALLOON_END    = pd.Timestamp("2023-02-25")

# Lead-length filter for sampling (must have a real lead)
MIN_LEAD_WORDS = 20
MAX_LEAD_WORDS = 100

In [ ]:
# =============================================================================
# CELL 2: Load corpus
# =============================================================================

CORPUS_PATH = RAW / "China_Fulltext_Merged_2020-2024.csv"
df = pd.read_csv(CORPUS_PATH)
df["pub_date"] = pd.to_datetime(df["pub_date"], utc=True).dt.tz_convert(None)

# Compute word counts for full_text (we'll need this later)
df["full_text_words"] = df["full_text"].fillna("").str.split().str.len()

print(f"Loaded corpus: {len(df)} articles")
print(f"\nDesk distribution (top 10):")
print(df["news_desk"].value_counts().head(10))
print(f"\nDate range: {df['pub_date'].min().date()} to {df['pub_date'].max().date()}")

In [ ]:

# =============================================================================
# CELL 3: Tokenization helper (used for entity matching)
# =============================================================================

# Simple alphanumeric tokenizer with hyphen preserved (e.g., "ride-hailing")
TOK_RE = re.compile(r"[A-Za-z][A-Za-z\-']+")
STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "is", "are", "was", "were", "be",
    "been", "being", "have", "has", "had", "do", "does", "did", "will",
    "would", "could", "should", "may", "might", "must", "can", "of", "to",
    "in", "on", "at", "by", "for", "with", "from", "as", "that", "this",
    "these", "those", "it", "its", "their", "they", "them", "we", "our",
    "us", "you", "your", "i", "my", "me", "he", "his", "him", "she", "her",
    "not", "no", "if", "than", "then", "so", "because", "while", "when",
    "where", "what", "which", "who", "whom", "how", "all", "any", "some",
    "more", "most", "other", "such", "only", "own", "same", "very", "just",
    "also", "still", "yet", "even", "now", "after", "before", "during",
    "between", "through", "over", "under", "above", "below", "out", "up",
    "down", "off", "into", "onto", "about",
}


def tokens(text):
    """Lowercase, alphanumeric content tokens, no stopwords."""
    if not isinstance(text, str):
        return []
    return [t.lower() for t in TOK_RE.findall(text)
            if t.lower() not in STOPWORDS and len(t) > 2]

In [ ]:

# =============================================================================
# CELL 4: Build distinctive-keyword lexicons (V4)
# Data-driven lexicon: words that appear much more in one desk than the other
# =============================================================================

def distinctive_keywords(corpus_a, corpus_b, top_n=150, min_df=3, max_df=0.85):
    """
    Find words that distinguish corpus_a from corpus_b.
    Uses log-odds ratio with informative Dirichlet prior (Monroe et al. 2008).
    """
    # Token frequency in each corpus
    a_tokens = [tok for text in corpus_a for tok in tokens(text)]
    b_tokens = [tok for text in corpus_b for tok in tokens(text)]
    a_count = Counter(a_tokens)
    b_count = Counter(b_tokens)

    # Document frequency for thresholding
    a_doc_count = Counter()
    for text in corpus_a:
        for tok in set(tokens(text)):
            a_doc_count[tok] += 1
    b_doc_count = Counter()
    for text in corpus_b:
        for tok in set(tokens(text)):
            b_doc_count[tok] += 1

    n_a_docs = len(corpus_a)
    n_b_docs = len(corpus_b)

    # All vocab
    all_words = set(a_count.keys()) | set(b_count.keys())
    a_total = sum(a_count.values())
    b_total = sum(b_count.values())

    results = []
    alpha = 0.01
    for w in all_words:
        # Document frequency filter
        a_df = a_doc_count.get(w, 0) / n_a_docs
        b_df = b_doc_count.get(w, 0) / n_b_docs
        if a_doc_count.get(w, 0) < min_df:
            continue
        if a_df > max_df:
            continue

        a_n = a_count.get(w, 0)
        b_n = b_count.get(w, 0)

        # Log-odds with Dirichlet prior
        odds_a = (a_n + alpha) / (a_total + alpha * len(all_words) - a_n - alpha)
        odds_b = (b_n + alpha) / (b_total + alpha * len(all_words) - b_n - alpha)
        log_odds = np.log(odds_a) - np.log(odds_b)

        # Variance (z-score adjustment)
        var = 1.0 / (a_n + alpha) + 1.0 / (b_n + alpha)
        z = log_odds / np.sqrt(var)

        results.append((w, log_odds, z, a_n, b_n))

    # Filter: only words MORE frequent in A
    results = [r for r in results if r[1] > 0]
    results.sort(key=lambda x: -x[2])  # by z-score
    return results[:top_n]


# Use leads (not full_text) for lexicon building — leads are tighter on topic entities
biz_leads = df[df["news_desk"] == "Business"]["lead_paragraph"].dropna().tolist()
foreign_leads = df[df["news_desk"] == "Foreign"]["lead_paragraph"].dropna().tolist()

print(f"Business leads: {len(biz_leads)}")
print(f"Foreign leads:  {len(foreign_leads)}")

print("\nExtracting BUSINESS_FRAME_V4...")
biz_distinctive = distinctive_keywords(biz_leads, foreign_leads, top_n=150)
BUSINESS_FRAME_V4 = {w for w, *_ in biz_distinctive}

print("Extracting WORLD_FRAME_V4...")
for_distinctive = distinctive_keywords(foreign_leads, biz_leads, top_n=150)
WORLD_FRAME_V4 = {w for w, *_ in for_distinctive}

# Remove any overlap (shouldn't happen but be safe)
overlap = BUSINESS_FRAME_V4 & WORLD_FRAME_V4
if overlap:
    BUSINESS_FRAME_V4 -= overlap
    WORLD_FRAME_V4 -= overlap
    print(f"  Removed {len(overlap)} overlapping words")

print(f"\nBUSINESS_FRAME_V4 ({len(BUSINESS_FRAME_V4)} words):")
print(sorted(BUSINESS_FRAME_V4)[:20], "...")
print(f"\nWORLD_FRAME_V4 ({len(WORLD_FRAME_V4)} words):")
print(sorted(WORLD_FRAME_V4)[:20], "...")

# Save lexicons
with open(PROCESSED / "lexicons.json", "w") as f:
    json.dump({
        "BUSINESS_FRAME_V4": sorted(BUSINESS_FRAME_V4),
        "WORLD_FRAME_V4": sorted(WORLD_FRAME_V4),
        "method": "Monroe et al. log-odds with Dirichlet prior, top 150 by z-score",
        "params": {"top_n": 150, "min_df": 3, "max_df": 0.85},
    }, f, indent=2)
print(f"\nSaved: {PROCESSED / 'lexicons.json'}")


In [ ]:

# =============================================================================
# CELL 5: Sample headlines (N_TOTAL_PER_DESK Business + N_TOTAL_PER_DESK Foreign)
# Expanded sample. The original 20 (sample_id 0–19) are drawn exactly as before
# (seed=SEED) and placed first, then extra headlines are appended from the
# remainder (seed=SEED_EXPAND). This keeps sample_id 0–19 stable so cached
# generations (keyed on sample_id) stay valid; only the new ids need generating.
# =============================================================================

# Eligibility filters
excluded_window = (
    ((df["pub_date"] >= SHANGHAI_START) & (df["pub_date"] <= SHANGHAI_END))
    | ((df["pub_date"] >= BALLOON_START) & (df["pub_date"] <= BALLOON_END))
)

eligible = df[
    df["lead_paragraph"].notna()
    & df["full_text"].notna()
    & (df["lead_paragraph"].str.split().str.len() >= MIN_LEAD_WORDS)
    & (df["lead_paragraph"].str.split().str.len() <= MAX_LEAD_WORDS)
    & ~excluded_window
].copy()

print(f"Eligible articles after filters: {len(eligible)}")
print(f"  By desk: {eligible['news_desk'].value_counts().head(5).to_dict()}")


def draw(desk):
    """Original N_ORIG locked rows (seed=SEED) + extra rows from the remainder."""
    pool = eligible[eligible["news_desk"] == desk]
    orig = pool.sample(n=N_ORIG_PER_DESK, random_state=SEED)  # exactly the original headlines
    extra = (pool.drop(orig.index)
                 .sample(n=N_TOTAL_PER_DESK - N_ORIG_PER_DESK, random_state=SEED_EXPAND))
    return orig, extra


biz_orig, biz_extra = draw("Business")
for_orig, for_extra = draw("Foreign")

# Order is critical: the original 20 must come first so sample_id 0–19 are
# unchanged. Otherwise cached generations (keyed on sample_id) would be reused
# for the wrong headline.
sampled = pd.concat([biz_orig, for_orig, biz_extra, for_extra]).reset_index(drop=True)
sampled["sample_id"] = range(len(sampled))

print(f"\nSampled {N_TOTAL_PER_DESK} Business + {N_TOTAL_PER_DESK} Foreign = {len(sampled)} headlines\n")
print("Business headlines:")
for _, r in sampled[sampled["news_desk"] == "Business"].iterrows():
    print(f"  [{r['pub_date'].date()}] {r['headline']}")
print("\nForeign headlines:")
for _, r in sampled[sampled["news_desk"] == "Foreign"].iterrows():
    print(f"  [{r['pub_date'].date()}] {r['headline']}")

# Save the sample
sample_cols = ["sample_id", "headline", "news_desk", "pub_date",
               "lead_paragraph", "full_text", "full_text_words"]
sampled[sample_cols].to_csv(PROCESSED / "experiment_sample.csv", index=False)
print(f"\nSaved: {PROCESSED / 'experiment_sample.csv'}")

In [ ]:

# =============================================================================
# CELL 6: Load DistilBERT for sentiment scoring
# =============================================================================

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
MAX_TOKENS_PER_CHUNK = 450
STRIDE = 50
BATCH_SIZE = 8

# Choose device: MPS > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, clean_up_tokenization_spaces=False)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
sentiment_model.to(device)
sentiment_model.eval()
print(f"Loaded model: {MODEL_NAME}")
print(f"Labels: {sentiment_model.config.id2label}")


def clear_memory():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


def make_token_chunks(text, max_tokens=MAX_TOKENS_PER_CHUNK, stride=STRIDE):
    """Tokenize once, split token IDs into overlapping chunks."""
    if not isinstance(text, str) or not text.strip():
        return []
    token_ids = tokenizer(
        text, add_special_tokens=False,
        truncation=False, return_attention_mask=False,
    )["input_ids"]
    if len(token_ids) == 0:
        return []
    chunks = []
    start = 0
    while start < len(token_ids):
        end = start + max_tokens
        chunk_ids = [tokenizer.cls_token_id] + token_ids[start:end] + [tokenizer.sep_token_id]
        chunks.append(chunk_ids)
        if end >= len(token_ids):
            break
        start = end - stride
    return chunks


def predict_token_chunks(token_id_chunks, batch_size=BATCH_SIZE):
    """Score pre-tokenized chunks. Returns DataFrame with sentiment per chunk."""
    if not token_id_chunks:
        return pd.DataFrame(columns=["chunk_id", "neg_prob", "pos_prob", "sentiment_score"])

    rows = []
    for start in range(0, len(token_id_chunks), batch_size):
        batch = token_id_chunks[start:start + batch_size]
        max_len = max(len(x) for x in batch)
        pad_id = tokenizer.pad_token_id

        input_ids, attn = [], []
        for ids in batch:
            pad_n = max_len - len(ids)
            input_ids.append(ids + [pad_id] * pad_n)
            attn.append([1] * len(ids) + [0] * pad_n)

        inputs = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long).to(device),
            "attention_mask": torch.tensor(attn, dtype=torch.long).to(device),
        }
        with torch.inference_mode():
            out = sentiment_model(**inputs)
            probs = torch.softmax(out.logits, dim=-1).detach().cpu().numpy()

        for j, prob in enumerate(probs):
            neg, pos = float(prob[0]), float(prob[1])
            rows.append({
                "chunk_id": start + j,
                "neg_prob": neg,
                "pos_prob": pos,
                "sentiment_score": pos - neg,
            })
        clear_memory()

    return pd.DataFrame(rows)


In [ ]:

# =============================================================================
# CELL 7: Score NYT real articles — entity axis (axis 1)
# =============================================================================

def entity_hit_rates(text):
    """Return (biz_rate, world_rate, biz_hits_n, world_hits_n)."""
    if not isinstance(text, str) or not text.strip():
        return 0.0, 0.0, 0, 0
    toks = tokens(text)
    if not toks:
        return 0.0, 0.0, 0, 0
    biz_n = sum(1 for t in toks if t in BUSINESS_FRAME_V4)
    world_n = sum(1 for t in toks if t in WORLD_FRAME_V4)
    return (
        100.0 * biz_n / len(toks),
        100.0 * world_n / len(toks),
        biz_n,
        world_n,
    )


axis1_rows = []
for _, r in sampled.iterrows():
    br, wr, bh, wh = entity_hit_rates(r["full_text"])
    axis1_rows.append({
        "sample_id": r["sample_id"],
        "news_desk": r["news_desk"],
        "source": "NYT (real)",
        "biz_rate": br,
        "world_rate": wr,
        "biz_hits_n": bh,
        "world_hits_n": wh,
        "word_count": r["full_text_words"],
    })

nyt_axis1 = pd.DataFrame(axis1_rows)
nyt_axis1.to_csv(PROCESSED / "nyt_baseline_axis1.csv", index=False)
print(f"Saved: {PROCESSED / 'nyt_baseline_axis1.csv'}")
print()
print("NYT entity-axis baseline (mean per desk):")
print(nyt_axis1.groupby("news_desk")[["biz_rate", "world_rate"]].mean().round(3))
print()
print("Fidelity scores (Business: biz-world; Foreign: world-biz):")
biz_sub = nyt_axis1[nyt_axis1["news_desk"] == "Business"]
for_sub = nyt_axis1[nyt_axis1["news_desk"] == "Foreign"]
print(f"  Business: {(biz_sub['biz_rate'] - biz_sub['world_rate']).mean():+.3f}")
print(f"  Foreign:  {(for_sub['world_rate'] - for_sub['biz_rate']).mean():+.3f}")


In [ ]:

# =============================================================================
# CELL 8: Score NYT real articles — sentiment axis (axis 2)
# =============================================================================

axis2_rows = []
for _, r in tqdm(sampled.iterrows(), total=len(sampled), desc="Scoring NYT"):
    try:
        chunks = make_token_chunks(r["full_text"])
        chunk_df = predict_token_chunks(chunks)
        if len(chunk_df) > 0:
            axis2_rows.append({
                "sample_id": r["sample_id"],
                "news_desk": r["news_desk"],
                "source": "NYT (real)",
                "sentiment": chunk_df["sentiment_score"].mean(),
                "n_chunks": len(chunk_df),
                "word_count": r["full_text_words"],
            })
    except Exception as e:
        print(f"Error on sid={r['sample_id']}: {e}")

nyt_axis2 = pd.DataFrame(axis2_rows)
nyt_axis2.to_csv(PROCESSED / "nyt_baseline_axis2.csv", index=False)
print(f"\nSaved: {PROCESSED / 'nyt_baseline_axis2.csv'}")
print()
print("NYT sentiment baseline:")
print(nyt_axis2.groupby("news_desk")["sentiment"].agg(["count", "mean", "std"]).round(4))
print()
biz_s = nyt_axis2[nyt_axis2["news_desk"] == "Business"]["sentiment"]
for_s = nyt_axis2[nyt_axis2["news_desk"] == "Foreign"]["sentiment"]
print(f"NYT Business mean:   {biz_s.mean():+.4f}")
print(f"NYT Foreign mean:    {for_s.mean():+.4f}")
print(f"Center (avg):        {(biz_s.mean() + for_s.mean()) / 2:+.4f}")
print(f"Gap (Biz - Foreign): {biz_s.mean() - for_s.mean():+.4f}")
print()
print("(Reference: paper baseline is Biz=-0.578, For=-0.445, gap=-0.133;"
      " these 20-sample numbers are consistent.)")

In [ ]:

# =============================================================================
# CELL 9: Summary of what was created
# =============================================================================

print("=" * 72)
print("NOTEBOOK 01 COMPLETE — Files written to data/processed/")
print("=" * 72)

for f in sorted(PROCESSED.glob("*")):
    size = f.stat().st_size
    print(f"  {f.name:35s} {size:>10,} bytes")

print(f"\n→ Notebook 02 will load these files for the GPT main experiment.")
print(f"→ Notebook 03 will load these files for the Gemini robustness probe.")
